# 5. Bajesovo učenje i Naivni Bajesov klasifikator

## Motivacija za probabilistički pristup

U praksi podaci **nisu savršeni** — šumoviti su, nepotpuni i ograničeni. Da bismo donosili
odluke u prisustvu **neizvesnosti**, potrebni su formalni modeli, a neizvesnost se formalno
modeluje **verovatnoćom**.

### Verovatnoća vs. stepen pripadnosti

| | verovatnoća $P(A)$ | stepen pripadnosti $\mu_A(x)$ |
|---|---|---|
| odnosi se na | **neizvesnost** | **neodređenost** |
| promenljiva $X$ | **nepoznata** | **poznata** |
| skup $A$ | dobro definisan | **neprecizno** definisan |
| pitanje | Kolika je verovatnoća da će se A dogoditi? | Koliko jako X pripada skupu A? |
| kada | **pre** događaja | **posle** događaja |

Naivni Bajes je **probabilistički** model — uvek radi **pre** donošenja odluke i računa
verovatnoće klasa.

## Bajesova teorema

$$P(h \mid D) = \frac{P(D \mid h) \cdot P(h)}{P(D)}$$

| član | naziv | značenje |
|---|---|---|
| $P(h \mid D)$ | **posterior** | verovatnoća hipoteze **nakon** što smo videli podatke |
| $P(D \mid h)$ | **likelihood** (verodostojnost) | koliko podaci podržavaju hipotezu |
| $P(h)$ | **prior** | početno verovanje, **pre** podataka |
| $P(D)$ | **evidence** | verovatnoća podataka; služi kao normalizacija |

> Bajesova teorema govori **kako da promenimo mišljenje kada dobijemo nove podatke**.
> Posterior je kompromis između podataka (likelihood) i prethodnog znanja (prior).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import load_iris, fetch_20newsgroups, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

plt.rcParams["figure.figsize"] = (9, 4.5); plt.rcParams["figure.dpi"] = 110
RS = 42

# --- primer sa predavanja: meningitis ---
P_M = 1 / 50_000       # ucestalost meningitisa u populaciji
P_S_M = 0.5            # ako neko ima meningitis, u 50% slucajeva ima ukocen vrat
P_S = 1 / 20           # ucestalost ukocenog vrata u opstoj populaciji

P_M_S = P_S_M * P_M / P_S
print(f"P(M)     = {P_M:.6f}   (prior — meningitis je REDAK)")
print(f"P(S|M)   = {P_S_M}      (likelihood)")
print(f"P(S)     = {P_S}       (evidence)")
print(f"\nP(M|S)   = {P_S_M} * {P_M:.6f} / {P_S} = {P_M_S:.6f} = {P_M_S*100:.2f}%")
print("\n-> Iako je ukocen vrat cest simptom, meningitis je izuzetno redak,")
print("   pa je konacna verovatnoca vrlo mala. SIMPTOM NIJE BOLEST.")
print("   Zato modeli koriste PRIOR — znanje koje imamo pre nego sto vidimo podatke.")

P(M)     = 0.000020   (prior — meningitis je REDAK)
P(S|M)   = 0.5      (likelihood)
P(S)     = 0.05       (evidence)

P(M|S)   = 0.5 * 0.000020 / 0.05 = 0.000200 = 0.02%

-> Iako je ukocen vrat cest simptom, meningitis je izuzetno redak,
   pa je konacna verovatnoca vrlo mala. SIMPTOM NIJE BOLEST.
   Zato modeli koriste PRIOR — znanje koje imamo pre nego sto vidimo podatke.


## Osnovna pravila verovatnoće

**Pravilo proizvoda (AND):**
$$P(A \cap B) = P(A \mid B)P(B) = P(B \mid A)P(A)$$
Ovo je osnova Bajesove teoreme.

**Pravilo zbira (OR):**
$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$
Presek se oduzima da se ne bi brojao dvaput.

**Zakon totalne verovatnoće:** ako su $A_1, \dots, A_n$ međusobno isključivi i pokrivaju ceo
univerzum ($\sum P(A_i) = 1$):
$$P(B) = \sum_{i=1}^{n} P(B \mid A_i) P(A_i)$$

U klasifikaciji su $A_i$ klase, $P(A_i)$ prior verovatnoće klasa, a njihov zbir je 1 —
zato modeli često **normalizuju** verovatnoće.

## Od MAP do Maximum Likelihood

**MAP** (Maximum A Posteriori) — biramo najverovatniju hipotezu nakon podataka. Pošto je
$P(D)$ konstanta:

$$h_{MAP} = \arg\max_{h \in \mathcal{H}} P(D \mid h) \, P(h)$$

**ML** (Maximum Likelihood) — ako pretpostavimo da su **sve hipoteze jednako verovatne**
(uniformni prior), prior ne utiče na izbor:

$$h_{ML} = \arg\max_{h \in \mathcal{H}} P(D \mid h)$$

> **MAP = podaci + prethodno znanje.  ML = samo podaci.**
> ML je specijalan slučaj MAP-a kada nemamo (ili ignorišemo) prior znanje.

## Bayes optimalni klasifikator

$$v^* = \arg\max_{v \in \mathcal{V}} \sum_{h \in \mathcal{H}} P(v \mid h) \, P(h \mid D)$$

To je **probabilističko glasanje**: svaki model glasa za neku klasu, glas se ponderiše
verovatnoćom tog modela, doprinosi se sabiraju.

**Zašto nije praktičan?** Zahteva razmatranje **svih** hipoteza. U realnim problemima broj
hipoteza je ogroman i računanje je neizvodljivo. Teorijski idealan, računski preskup.

In [2]:
# primer Bayes-optimalne odluke sa predavanja
hipoteze = {"h1": 0.5, "h2": 0.3, "h3": 0.4}      # posterior P(h|D)
odluke   = {"h1": "+", "h2": "-", "h3": "-"}       # sta svaka hipoteza predvidja

glasovi = {"+": 0.0, "-": 0.0}
for h, p in hipoteze.items():
    glasovi[odluke[h]] += p

for h in hipoteze:
    print(f"   {h}: P(h|D) = {hipoteze[h]}, predvidja '{odluke[h]}'")
print(f"\nUkupna podrska:  '+' = {glasovi['+']:.1f}   '-' = {glasovi['-']:.1f}")
print(f"Bayes-optimalna odluka: '{max(glasovi, key=glasovi.get)}'")
print("\n-> h1 je najverovatnija pojedinacna hipoteza i glasa za '+',")
print("   ali h2 i h3 ZAJEDNO imaju vecu podrsku za '-'.")

   h1: P(h|D) = 0.5, predvidja '+'
   h2: P(h|D) = 0.3, predvidja '-'
   h3: P(h|D) = 0.4, predvidja '-'

Ukupna podrska:  '+' = 0.5   '-' = 0.7
Bayes-optimalna odluka: '-'

-> h1 je najverovatnija pojedinacna hipoteza i glasa za '+',
   ali h2 i h3 ZAJEDNO imaju vecu podrsku za '-'.


## Naivna pretpostavka

Pošto je Bayes-optimalno preskupo, pravimo **svesno pojednostavljenje**.

> **Naivna pretpostavka:** atributi su **uslovno nezavisni** data klasa.
> $$a_1 \perp a_2 \perp \dots \perp a_n \mid v_j$$

Ne tvrdimo da su atributi u stvarnosti nezavisni — tvrdimo samo da ih **model tako tretira**.

### Matematička posledica

Bez pretpostavke, $P(a_1, a_2, \dots, a_n \mid v_j)$ je složena zajednička verovatnoća.
Sa pretpostavkom:

$$P(a_1, a_2, \dots, a_n \mid v_j) = \prod_{i=1}^{n} P(a_i \mid v_j)$$

Umesto jedne komplikovane verovatnoće računamo mnoštvo jednostavnih i **množimo ih**.
Zato Naive Bayes radi i sa 500+ atributa.

### Konačna formula

$$v^* = \arg\max_{v_j \in \mathcal{V}} P(v_j) \prod_{i=1}^{n} P(a_i \mid v_j)$$

Iz podataka se procenjuju samo: **prior klasa** $P(v_j)$ i **uslovne verovatnoće** $P(a_i \mid v_j)$.

## Problem nulte verovatnoće i Laplasovo poravnanje

Ako je za neki atribut $P(a_i \mid v_j) = 0$, ceo proizvod postaje **nula** i klasa se
eliminiše bez obzira na sve ostale atribute. Sa 500 atributa dovoljna je **jedna** nula.

**Rešenje — Laplasovo poravnanje** (Laplace smoothing):

$$\hat{P}(a_i \mid v_j) = \frac{C(a_i, v_j) + m \cdot P}{C(v_j) + m}$$

- $C(a_i, v_j)$ — broj pojavljivanja atributa u klasi
- $C(v_j)$ — ukupan broj uzoraka klase
- $P$ — prior procena verovatnoće atributa
- $m$ — težina priora (parametar poravnanja)

Ne tvrdimo da je verovatnoća nula, već da je **mala ali ne nula**. To čini model robustnim.

### Primer sa predavanja — Car Theft

In [3]:
auta = pd.DataFrame({
    "Color":  ["Red","Red","Red","Yellow","Yellow","Yellow","Yellow","Yellow","Red","Red"],
    "Type":   ["Sports","Sports","Sports","Sports","Sports","SUV","SUV","SUV","SUV","Sports"],
    "Origin": ["Domestic","Domestic","Domestic","Domestic","Imported","Imported","Imported",
               "Domestic","Imported","Imported"],
    "Stolen": ["Y","N","Y","N","Y","N","Y","N","N","Y"],
})
display(auta)

nY = (auta.Stolen == "Y").sum(); nN = (auta.Stolen == "N").sum()
print(f"\nn_Y = {nY}, n_N = {nN}   ->   P(Y) = {nY/len(auta)}, P(N) = {nN/len(auta)}")
print("\nPitanje: da li ce 'Red Domestic SUV' biti ukraden?")
print("(ta kombinacija NE postoji u tabeli — zato nam treba model)")

,Color,Type,Origin,Stolen
0,Red,Sports,Domestic,Y
1,Red,Sports,Domestic,N
2,Red,Sports,Domestic,Y
3,Yellow,Sports,Domestic,N
4,Yellow,Sports,Imported,Y
5,Yellow,SUV,Imported,N
6,Yellow,SUV,Imported,Y
7,Yellow,SUV,Domestic,N
8,Red,SUV,Imported,N
9,Red,Sports,Imported,Y



n_Y = 5, n_N = 5   ->   P(Y) = 0.5, P(N) = 0.5

Pitanje: da li ce 'Red Domestic SUV' biti ukraden?
(ta kombinacija NE postoji u tabeli — zato nam treba model)


In [4]:
m_par, P_par = 3, 0.5          # parametri poravnanja sa predavanja

def laplas(atribut, vrednost, klasa):
    'P_hat(a_i | v_j) sa Laplasovim poravnanjem.'
    C_ai_vj = ((auta[atribut] == vrednost) & (auta.Stolen == klasa)).sum()
    C_vj = (auta.Stolen == klasa).sum()
    return (C_ai_vj + m_par * P_par) / (C_vj + m_par)

upit = {"Color": "Red", "Type": "SUV", "Origin": "Domestic"}
print(f"Imenilac je svuda isti: C(v_j) + m = 5 + 3 = 8\n")

verovatnoce = {}
for klasa in ("Y", "N"):
    print(f"Klasa {klasa}:")
    for atr, vred in upit.items():
        p = laplas(atr, vred, klasa)
        broj = ((auta[atr] == vred) & (auta.Stolen == klasa)).sum()
        print(f"   P({vred:<9}|{klasa}) = ({broj} + 3*0.5)/8 = {p:.4f}")
        verovatnoce[(atr, klasa)] = p
    print()

skor = {}
for klasa in ("Y", "N"):
    s = 0.5                                      # P(v_j)
    for atr in upit:
        s *= verovatnoce[(atr, klasa)]
    skor[klasa] = s
    print(f"Score({klasa}) = 0.5 * " + " * ".join(f"{verovatnoce[(a,klasa)]:.2f}" for a in upit)
          + f" = {s:.5f}")

odluka = max(skor, key=skor.get)
print(f"\nScore(N) {'>' if skor['N']>skor['Y'] else '<'} Score(Y)  ->  PREDIKCIJA: {odluka}")
print("   (auto NECE biti ukraden)" if odluka == "N" else "   (auto CE biti ukraden)")

Imenilac je svuda isti: C(v_j) + m = 5 + 3 = 8

Klasa Y:
   P(Red      |Y) = (3 + 3*0.5)/8 = 0.5625
   P(SUV      |Y) = (1 + 3*0.5)/8 = 0.3125
   P(Domestic |Y) = (2 + 3*0.5)/8 = 0.4375

Klasa N:
   P(Red      |N) = (2 + 3*0.5)/8 = 0.4375
   P(SUV      |N) = (3 + 3*0.5)/8 = 0.5625
   P(Domestic |N) = (3 + 3*0.5)/8 = 0.5625

Score(Y) = 0.5 * 0.56 * 0.31 * 0.44 = 0.03845
Score(N) = 0.5 * 0.44 * 0.56 * 0.56 = 0.06921

Score(N) > Score(Y)  ->  PREDIKCIJA: N
   (auto NECE biti ukraden)


## Varijante u `scikit-learn`

| klasa | pretpostavka o $P(a_i \mid v_j)$ | kada se koristi |
|---|---|---|
| `GaussianNB` | normalna raspodela | **neprekidni** atributi |
| `MultinomialNB` | multinomijalna (broj pojavljivanja) | **tekst**, brojanje reči |
| `BernoulliNB` | Bernoulli (0/1) | binarni atributi |

In [5]:
iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=0.3,
                                      random_state=RS, stratify=iris.target)
gnb = GaussianNB().fit(Xtr, ytr)
print(f"GaussianNB na iris: tacnost {gnb.score(Xte, yte):.4f}")
print(f"\nNauceni prior klasa: {gnb.class_prior_.round(3)}")
print(f"Nauceni proseci po klasi (prve 2 kolone):\n{gnb.theta_[:, :2].round(2)}")

GaussianNB na iris: tacnost 0.9111

Nauceni prior klasa: [0.333 0.333 0.333]
Nauceni proseci po klasi (prve 2 kolone):
[[4.99 3.43]
 [5.95 2.73]
 [6.68 3.01]]


In [6]:
# MultinomialNB na tekstu — zadatak za koji je Naive Bayes i napravljen
kat = ["rec.sport.hockey", "sci.space", "talk.politics.mideast"]
tr = fetch_20newsgroups(subset="train", categories=kat, remove=("headers","footers","quotes"))
te = fetch_20newsgroups(subset="test", categories=kat, remove=("headers","footers","quotes"))

vec = CountVectorizer(stop_words="english", min_df=2)
Xtr_t = vec.fit_transform(tr.data)            # fit SAMO na treningu
Xte_t = vec.transform(te.data)

mnb = MultinomialNB().fit(Xtr_t, tr.target)
print(f"Dokumenata: {len(tr.data)} trening, {len(te.data)} test")
print(f"Recnik: {len(vec.vocabulary_):,} reci")
print(f"\nTacnost MultinomialNB: {accuracy_score(te.target, mnb.predict(Xte_t)):.4f}")
print(f"Bazna linija (najcesca klasa): {np.bincount(te.target).max()/len(te.target):.4f}")

Dokumenata: 1757 trening, 1169 test
Recnik: 13,983 reci

Tacnost MultinomialNB: 0.9170
Bazna linija (najcesca klasa): 0.3413


## Kada naivna pretpostavka škodi

Naive Bayes radi **iznenađujuće dobro** kada je pretpostavka približno tačna (tekst,
gde je vreća reči prirodna reprezentacija).

Ali kada su atributi **jako korelisani**, model ih broji višestruko i rezultat pada.
Sledeći primer to pokazuje merenjem.

In [7]:
from sklearn.linear_model import LogisticRegression

print(f"{'situacija':<34} {'GaussianNB':>11} {'LogReg':>9}")
print("-" * 56)
for opis, redundantnih in [("nezavisni atributi (0 kopija)", 0),
                           ("umereno korelisani (5 kopija)", 5),
                           ("jako korelisani (15 kopija)", 15)]:
    Xc, yc = make_classification(n_samples=1200, n_features=20, n_informative=5,
                                 n_redundant=redundantnih, random_state=RS)
    a = cross_val_score(GaussianNB(), Xc, yc, cv=5).mean()
    b = cross_val_score(LogisticRegression(max_iter=3000), Xc, yc, cv=5).mean()
    print(f"{opis:<34} {a:>11.4f} {b:>9.4f}")
print("\n-> Kako raste korelacija, NB gubi vise od logisticke regresije,")
print("   jer krsi sopstvenu pretpostavku o uslovnoj nezavisnosti.")

situacija                           GaussianNB    LogReg
--------------------------------------------------------
nezavisni atributi (0 kopija)           0.8033    0.7358
umereno korelisani (5 kopija)           0.8208    0.7267
jako korelisani (15 kopija)             0.7825    0.7367

-> Kako raste korelacija, NB gubi vise od logisticke regresije,
   jer krsi sopstvenu pretpostavku o uslovnoj nezavisnosti.


## Sažetak

**Prednosti**
- vrlo brz, radi i sa 500+ atributa
- traži malo podataka za procenu parametara
- prirodno daje verovatnoće klasa
- odličan baseline za klasifikaciju teksta

**Nedostaci**
- pretpostavka uslovne nezavisnosti je u praksi skoro uvek narušena
- problem nulte verovatnoće (rešava se Laplasovim poravnanjem)
- procenjene verovatnoće su često loše kalibrisane (previše samouverene)